# UC-MF-1 — Trajectory Basics via OGC API - Moving Features

**Persona:** Logistics Analyst

**Goal:** Create a moving feature representing a freight vehicle, attach a temporal
geometry sequence (the trajectory), query the sequence back, then open the
Moving Features browser for visual inspection.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `moving_features` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)

**References:**
- OGC API — Moving Features 1.0 (OGC 21-017r1)
- Routes: `/movingfeatures/catalogs/{cat}/collections/{col}/items`
  and `…/items/{mf_id}/tgsequence`

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-logistics")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "freight-vehicles")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform  : {BASE_URL}")
print(f"Catalog   : {CATALOG_ID}")
print(f"Collection: {COLLECTION_ID}")

## Step 1 — Create catalog and collection

Moving Features uses the standard DynaStore catalog/collection hierarchy.
We create a demo catalog and a `freight-vehicles` collection via the OGC Features
catalog management API. Existing resources return 409 and are skipped.

In [ ]:
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "Logistics demo"}),
)
print(f"catalog  : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({"id": COLLECTION_ID, "title": "Freight vehicle trajectories"}),
)
print(f"collection: {r.status_code}", "(already exists)" if r.status_code == 409 else "")

## Step 2 — Create a moving feature

`POST /movingfeatures/catalogs/{cat}/collections/{col}/items` with a
`MovingFeatureCreate` body (JSON with `feature_type` and optional `properties`).
The server returns a `MovingFeature` with a stable `id` (UUID).

In [ ]:
mf_payload = {
    "feature_type": "Feature",
    "properties": {
        "vehicle_id": "TRK-001",
        "carrier": "Demo Logistics SpA",
        "cargo": "agricultural inputs",
    },
}

r = client.post(
    f"/movingfeatures/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    content=json.dumps(mf_payload),
)
print(f"create moving feature: {r.status_code}")

_mf_id = None
if r.status_code == 201:
    mf = r.json()
    _mf_id = mf["id"]
    print(f"  id           : {_mf_id}")
    print(f"  catalog_id   : {mf.get('catalog_id')}")
    print(f"  collection_id: {mf.get('collection_id')}")
    print(f"  created_at   : {mf.get('created_at')}")
else:
    print(r.text[:400])

## Step 3 — Attach a temporal geometry sequence (trajectory)

`POST /movingfeatures/catalogs/{cat}/collections/{col}/items/{mf_id}/tgsequence`
stores a `TemporalGeometryCreate` payload: ordered datetimes + matching coordinate
array. This represents the vehicle route from Rome to Naples.

In [ ]:
if _mf_id is None:
    print("SKIP: no moving feature created in step 2 — cannot add trajectory.")
else:
    tg_payload = {
        "datetimes": [
            "2024-03-15T06:00:00Z",
            "2024-03-15T07:30:00Z",
            "2024-03-15T09:00:00Z",
            "2024-03-15T10:45:00Z",
        ],
        "coordinates": [
            [12.4964, 41.9028],  # Rome — Termini area
            [13.3360, 41.4865],  # Frosinone
            [13.7988, 41.1209],  # Cassino
            [14.2681, 40.9028],  # Naples — outskirts
        ],
        "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84",
        "trs": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian",
        "interpolation": "Linear",
        "properties": {
            "speed_kmh": [90, 95, 100, 85],
            "heading_deg": [160, 165, 155, 150],
        },
    }

    r = client.post(
        f"/movingfeatures/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{_mf_id}/tgsequence",
        content=json.dumps(tg_payload),
    )
    print(f"add tgsequence: {r.status_code}")
    if r.status_code == 201:
        tg = r.json()
        _tg_id = tg["id"]
        print(f"  tg.id          : {_tg_id}")
        print(f"  tg.mf_id       : {tg.get('mf_id')}")
        print(f"  tg.interpolation: {tg.get('interpolation')}")
        print(f"  datetimes      : {tg.get('datetimes')}")
    else:
        print(r.text[:400])

## Step 4 — List moving features in the collection

`GET /movingfeatures/catalogs/{cat}/collections/{col}/items` returns the list
of moving features. Confirms the feature we created is visible.

In [ ]:
r = client.get(
    f"/movingfeatures/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"limit": 10},
)
print(f"list items: {r.status_code}")
if r.status_code == 200:
    items = r.json()
    if isinstance(items, list):
        print(f"Count: {len(items)}")
        for it in items[:5]:
            props = it.get("properties") or {}
            print(f"  id={it.get('id')}  vehicle={props.get('vehicle_id', 'n/a')}")
    else:
        print(json.dumps(items, indent=2)[:400])
else:
    print(r.text[:300])

## Step 5 — Read back the temporal geometry sequence

`GET /movingfeatures/catalogs/{cat}/collections/{col}/items/{mf_id}/tgsequence`
returns all temporal geometry sequences attached to the moving feature.

In [ ]:
if _mf_id is None:
    print("SKIP: no moving feature created — skipping tgsequence read-back.")
else:
    r = client.get(
        f"/movingfeatures/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{_mf_id}/tgsequence"
    )
    print(f"get tgsequence: {r.status_code}")
    if r.status_code == 200:
        seqs = r.json()
        if isinstance(seqs, list):
            print(f"Sequences: {len(seqs)}")
            for seq in seqs:
                print(f"  id={seq.get('id')}")
                print(f"  datetimes : {seq.get('datetimes')}")
                print(f"  coordinates: {seq.get('coordinates')}")
                print(f"  interpolation: {seq.get('interpolation')}")
        else:
            print(json.dumps(seqs, indent=2)[:400])
    else:
        print(r.text[:300])

## Teardown

Delete the moving feature and demo catalog. The `DELETE` on the moving feature
cascades and removes temporal geometry sequences. Set `MF_TEARDOWN=false` to
keep the data for further inspection.

In [ ]:
_TEARDOWN = os.environ.get("MF_TEARDOWN", "true").lower() == "true"

if _TEARDOWN and _mf_id:
    r = client.delete(
        f"/movingfeatures/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{_mf_id}"
    )
    print(f"DELETE moving feature: {r.status_code}")

    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
    print(f"DELETE collection    : {r.status_code}")

    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog       : {r.status_code}")
elif not _TEARDOWN:
    print("Teardown skipped (set MF_TEARDOWN=true to enable).")
else:
    print("Teardown skipped: no moving feature was created.")

## Result — Open the Moving Features browser

In [ ]:
client.close()
print(f"Open the Moving Features browser: {BASE_URL}/web/pages/movingfeatures_browser?language=en")